# CDM scientific comparison

## Context and methods
Compare the current product with corrected reference B, which is generated from
frozen C source without importing C or ITAMAE products. B independently changes
the gravitational constant and inverts the NFW mass function using the
principal Lambert-W branch at 50-digit precision. Full one-factor records and
A/B regeneration live in `sashimi-family/validation/references/sashimi-c`.

Run from the C repository root with the reviewed candidate wheels installed.
`tests/references` is a validation input, not a runtime dependency.

### Assumptions
The Planck-calibrated background, Yang accretion, concentration scatter,
`pert2_shanks`, `ct_th=0` and all grid values are fixed by the sidecar. Agreement
validates migration under this specification; it does not establish simulation
calibration or convergence of every observable.

In [ ]:
import hashlib, json
from pathlib import Path
import numpy as np
from sashimi_c import SubhaloProperties, diagnose_stripping_approximation
bundle = Path.cwd()
if not (bundle / 'tests/references').is_dir():
    bundle = bundle.parent
reference_path = bundle / 'tests/references/B-all.npz'
record = json.loads(reference_path.with_suffix('.json').read_text())
assert hashlib.sha256(reference_path.read_bytes()).hexdigest() == record['artifact_sha256']
assert record['role'] == 'B'
parameters = record['calculation']['parameters']
print('B source:', record['source_revision'])
print('parameters:', parameters)

## Results: all catalog columns
The acceptance tolerance is 5e-12 for this independent B/C calculation. It
covers roundoff, interpolation evaluation and the independent inverse solver;
it is tighter than the preserved historical full-observable regression.

In [ ]:
model = SubhaloProperties()
actual = model.subhalo_properties_calc(**parameters)
with np.load(reference_path) as reference:
    differences = {}
    for index, values in enumerate(actual):
        target = reference[f'tuple_{index}']
        if values.dtype.kind == 'b':
            np.testing.assert_array_equal(values, target)
            differences[f'tuple_{index}'] = int(np.count_nonzero(values != target))
        else:
            np.testing.assert_allclose(values, target, rtol=5e-12, atol=0.)
            differences[f'tuple_{index}'] = float(np.max(abs(values-target)/np.maximum(abs(target), 1e-300)))
print(differences)
print(dict(model.catalog.metadata))

## Solver diagnostic
The perturbative default is compared to direct ODE integration. This reports
an approximation error; it does not automatically select Picard or ODE as a new
default. Dedicated grid/solver convergence artifacts are reviewed separately.

In [ ]:
diagnostic = diagnose_stripping_approximation(1e12, np.logspace(6., 10., 9), accretion_redshift=1.)
print(dict(diagnostic.summary()))
assert np.all(np.isfinite(diagnostic.relative_difference))

## Takeaways
The executed cells test every B/C catalog entry and report the direct ODE
comparison under fixed conditions. Historical fixture provenance remains
unchanged. Full convergence and prompt-cusp input limitations are tracked in
the release review record; a successful regression is not a physical calibration.

## Saved one-variable resolution sweep

The hash-verified summary comes from complete saved catalogs, with original
source SHAs and any failure/warning records retained. Baseline M0=1e12 Msun,
zmax=3, N_ma=16, dz=.25, N_herm=3, N_hermNa=8, ct_th=.77 and pert2_shanks
are fixed unless the plotted axis changes. Accretion M200 spans 1e6–1e10 Msun.
The finest grid is a comparator, not continuum truth or a universal error bound.

In [ ]:
import hashlib
import matplotlib.pyplot as plt
science_dir = bundle / "validation/science"
summary_file = science_dir / "convergence-summary.json"
provenance = json.loads((science_dir / "provenance.json").read_text())
assert hashlib.sha256(summary_file.read_bytes()).hexdigest() == provenance["summary_sha256"]
summary = json.loads(summary_file.read_text())
state = "default" if summary["variant"] == "sashimi-c" else "sidm"
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis, parameter, base_value in zip(axes, ["N_ma", "dz"], [16, .25]):
    selected = [(base_value, summary["rows"]["baseline"])]
    selected += [(row["parameters"][parameter], row) for label, row in summary["rows"].items() if label.startswith(parameter + "-")]
    selected.sort(key=lambda pair: pair[0], reverse=parameter == "dz")
    values = np.array([row["states"][state]["metrics"]["bound_mass_fraction"] for _, row in selected])
    axis.semilogx([x for x, _ in selected], (values / values[-1] - 1) * 100, "o-")
    axis.set(xlabel="Mass nodes N_ma" if parameter == "N_ma" else "Redshift step dz", ylabel="Bound mass fraction difference from finest [%]")
    if parameter == "dz": axis.invert_xaxis()
    axis.grid(alpha=.2)
fig.suptitle("Finite-grid sensitivity at fixed reduced settings")
fig.tight_layout()
plt.show()
for check in summary["final_refinements"]:
    print(check["before"], "→", check["after"], "bound mass change [%]", format(check["relative_percent"]["bound_mass_fraction"], ".6g"))

The last mass-node and redshift refinements still change bound mass fractions
by about 0.50% and 0.75–0.77%, respectively. At the coarse baseline, switching
pert2_shanks to direct ODE changes bound mass by about -3.82%; tightening ODE
tolerances has a much smaller effect. These distinctions remain visible and do
not change the product defaults. See `docs/resolution-and-states.md` for the
full table, original EPS failure and separate post-fix successful controls.